In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("dair-ai/emotion")

print(dataset)

c:\Users\sondo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [2]:
print("Train distribution:")
print(dataset["train"].to_pandas()["label"].value_counts().sort_index())

print("\nValidation distribution:")
print(dataset["validation"].to_pandas()["label"].value_counts().sort_index())

print("\nTest distribution:")
print(dataset["test"].to_pandas()["label"].value_counts().sort_index())

Train distribution:
label
0    4666
1    5362
2    1304
3    2159
4    1937
5     572
Name: count, dtype: int64

Validation distribution:
label
0    550
1    704
2    178
3    275
4    212
5     81
Name: count, dtype: int64

Test distribution:
label
0    581
1    695
2    159
3    275
4    224
5     66
Name: count, dtype: int64


In [3]:
print(dataset["train"].features["label"])


ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])


In [4]:
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

print(train_df.head())
print("\nMissing values:")
print(train_df.isnull().sum())

                                                text  label
0                            i didnt feel humiliated      0
1  i can go from feeling so hopeless to so damned...      0
2   im grabbing a minute to post i feel greedy wrong      3
3  i am ever feeling nostalgic about the fireplac...      2
4                               i am feeling grouchy      3

Missing values:
text     0
label    0
dtype: int64


In [5]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

for df in [train_df, val_df, test_df]:
    df["clean_text"] = df["text"].apply(clean_text)

print(train_df[["text", "clean_text"]].head())

                                                text  \
0                            i didnt feel humiliated   
1  i can go from feeling so hopeless to so damned...   
2   im grabbing a minute to post i feel greedy wrong   
3  i am ever feeling nostalgic about the fireplac...   
4                               i am feeling grouchy   

                                          clean_text  
0                            i didnt feel humiliated  
1  i can go from feeling so hopeless to so damned...  
2   im grabbing a minute to post i feel greedy wrong  
3  i am ever feeling nostalgic about the fireplac...  
4                               i am feeling grouchy  


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_df["clean_text"])
X_val = vectorizer.transform(val_df["clean_text"])
X_test = vectorizer.transform(test_df["clean_text"])

y_train = train_df["label"]
y_val = val_df["label"]
y_test = test_df["label"]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (16000, 30000)
X_val: (2000, 30000)
X_test: (2000, 30000)


In [7]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

baseline_model.fit(X_train, y_train)

print("Baseline model trained successfully.")

Baseline model trained successfully.


In [8]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

val_predictions = baseline_model.predict(X_val)

val_accuracy = accuracy_score(y_val, val_predictions)
val_macro_f1 = f1_score(
    y_val,
    val_predictions,
    average="macro"
)

print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Macro F1: {val_macro_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_predictions,
        target_names=[
            "sadness",
            "joy",
            "love",
            "anger",
            "fear",
            "surprise"
        ]
    )
)

Validation Accuracy: 0.8765
Validation Macro F1: 0.8498

Classification Report:
              precision    recall  f1-score   support

     sadness       0.92      0.88      0.90       550
         joy       0.92      0.88      0.90       704
        love       0.75      0.93      0.83       178
       anger       0.88      0.89      0.88       275
        fear       0.82      0.82      0.82       212
    surprise       0.73      0.80      0.76        81

    accuracy                           0.88      2000
   macro avg       0.84      0.87      0.85      2000
weighted avg       0.88      0.88      0.88      2000



In [9]:
test_predictions = baseline_model.predict(X_test)

test_accuracy = accuracy_score(y_test, test_predictions)
test_macro_f1 = f1_score(
    y_test,
    test_predictions,
    average="macro"
)

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Macro F1: {test_macro_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=[
            "sadness",
            "joy",
            "love",
            "anger",
            "fear",
            "surprise"
        ]
    )
)

Test Accuracy: 0.8685
Test Macro F1: 0.8301

Classification Report:
              precision    recall  f1-score   support

     sadness       0.93      0.87      0.90       581
         joy       0.91      0.87      0.89       695
        love       0.70      0.90      0.79       159
       anger       0.86      0.89      0.87       275
        fear       0.87      0.82      0.84       224
    surprise       0.59      0.82      0.69        66

    accuracy                           0.87      2000
   macro avg       0.81      0.86      0.83      2000
weighted avg       0.88      0.87      0.87      2000

